# collection.ipynb

Pull B-class / C-class English Wikipedia articles for DeBERTa-WP-BCLASS.

## Setup: get title lists from Quarry

1. Go to https://quarry.wmcloud.org and log in.
2. New Query, database `enwiki_p`, and run the query in `quarry_b.sql`.
3. Download the result as CSV, save as `b_class_titles.csv` next to this notebook.
4. Run the query in `quarry_c.sql` and save as `c_class_titles.csv`.

## Labeling rules

- **C-class**: requires a fully parsed b1-b6 checklist on the talk page. Rows without a complete checklist are skipped.
- **B-class**: checklist is fetched opportunistically, not required. If no checklist is found (or it's incomplete), the row is kept with all six criteria assumed `yes`. If a *complete* checklist IS found and at least one of the six is not `yes`, the row is **skipped**.

In [1]:
import csv
import json
import os
import re
import time
import requests
import random
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Config
N_PER_CLASS = 1000
OUT_PATH = "bclass_data.jsonl"
DELAY = 1.0  # seconds between requests

WIKIMEDIA_TOKEN = os.environ.get("WIKIMEDIA_TOKEN")  # optional, enables LiftWing scoring

API = "https://en.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": "bclass-research-bot/0.1 (research use; contact: jake) requests"
}
LIFTWING_URL = "https://api.wikimedia.org/service/lw/inference/v1/models/enwiki-articlequality:predict"

B_PARAM_RE = re.compile(r"\|\s*b([1-6])\s*=\s*([a-zA-Z]*)", re.IGNORECASE)
B_KEYS = ["b1", "b2", "b3", "b4", "b5", "b6"]

print(f"N_PER_CLASS={N_PER_CLASS}  OUT_PATH={OUT_PATH}  DELAY={DELAY}  LiftWing={'on' if WIKIMEDIA_TOKEN else 'off'}")


N_PER_CLASS=1000  OUT_PATH=bclass_data.jsonl  DELAY=1.0  LiftWing=on


In [3]:
def api_get(params, max_attempts=6):
    """GET against the Action API. Prints every request's status code.
    On 429, honors the Retry-After header rather than guessing at a backoff."""
    params = {**params, "format": "json"}
    label = params.get("titles", params.get("action", "?"))
    for attempt in range(max_attempts):
        r = requests.get(API, params=params, headers=HEADERS, timeout=30)
        print(f"    [http {r.status_code}] {label}")
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 5 * (attempt + 1)))
            print(f"    [429] rate-limited, waiting {wait}s (attempt {attempt + 1}/{max_attempts})")
            time.sleep(wait)
            continue
        print(f"    [error {r.status_code}] attempt {attempt + 1}/{max_attempts}, backing off")
        time.sleep(2 * (attempt + 1))
    r.raise_for_status()


def get_wikitext(title):
    data = api_get({
        "action": "query",
        "prop": "revisions",
        "titles": title,
        "rvslots": "main",
        "rvprop": "content|timestamp",
    })
    pages = data.get("query", {}).get("pages", {})
    for p in pages.values():
        revs = p.get("revisions")
        if not revs:
            return None, None
        slot = revs[0]["slots"]["main"]
        return slot.get("*", ""), revs[0].get("timestamp")
    return None, None


def get_plaintext(title):
    data = api_get({
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "titles": title,
    })
    pages = data.get("query", {}).get("pages", {})
    for p in pages.values():
        return p.get("extract", "")
    return ""


def extract_b_flags(wikitext):
    """Return (dict b1..b6 -> normalized value, conflict_bool).
    Returns (None, conflict_bool) if the checklist isn't fully present
    (fewer than 6 well-formed b1..b6 values found)."""
    matches = B_PARAM_RE.findall(wikitext or "")
    if not matches:
        return None, False

    seen = {}
    conflict = False
    for num, val in matches:
        val_norm = val.strip().lower()
        if val_norm in ("y", "yes"):
            val_norm = "yes"
        elif val_norm in ("n", "no"):
            val_norm = "no"
        elif val_norm in ("na", "n/a"):
            val_norm = "na"
        elif val_norm == "":
            continue
        else:
            continue  # junk value like "gobbledygook" -> skip

        key = f"b{num}"
        if key in seen and seen[key] != val_norm:
            conflict = True
        seen[key] = val_norm

    if len(seen) < 6:
        return None, conflict  # incomplete checklist
    return seen, conflict


def get_latest_revid(title):
    data = api_get({"action": "query", "prop": "info", "titles": title})
    pages = data.get("query", {}).get("pages", {})
    for p in pages.values():
        return p.get("lastrevid")
    return None


def query_liftwing(revid):
    """Return dict of class->probability from LiftWing's article quality
    model, or None on failure. Requires WIKIMEDIA_TOKEN to be set."""
    if not WIKIMEDIA_TOKEN or revid is None:
        return None
    try:
        r = requests.post(
            LIFTWING_URL,
            json={"rev_id": revid},
            headers={**HEADERS, "Authorization": f"Bearer {WIKIMEDIA_TOKEN}"},
            timeout=30,
        )
        print(f"    [liftwing http {r.status_code}] revid={revid}")
        if r.status_code != 200:
            return None
        data = r.json()
        return data.get("enwiki", {}).get("scores", {}).get(str(revid), {}) \
            .get("articlequality", {}).get("score", {}).get("probability")
    except requests.RequestException as e:
        print(f"    [liftwing error] {e}")
        return None


def load_existing_titles(out_path, cls):
    """Read out_path (if it exists) and return the set of article titles
    already collected for this class -- lets a rerun resume instead of
    redoing completed work."""
    done = set()
    if not os.path.exists(out_path):
        return done
    with open(out_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            if row.get("overall_class") == cls:
                done.add(row.get("article_title"))
    return done


def load_quarry_titles(csv_path, limit, seed=None):
    titles = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            t = row.get("page_title") or row.get("title")
            if t:
                titles.append(t.replace("_", " "))
    random.Random(seed).shuffle(titles)
    return titles[:limit]


def build_dataset(cls, article_titles, n, delay=DELAY, out_path=None):
    """
    article_titles: list of ARTICLE titles (namespace 0, from Quarry)
    "Talk:" is prepended per-title below.

    cls == "C": checklist is REQUIRED. Rows without a complete b1..b6
                 block are skipped.
    cls == "B": checklist is fetched OPPORTUNISTICALLY, not required.
                 - No checklist / incomplete checklist -> keep the row,
                   assume all six criteria = 'yes' (that's what class=B
                   means by definition).
                 - Complete checklist found, all six = 'yes' -> keep,
                   use the real (all-yes) flags.
                 - Complete checklist found, but at least one flag is
                   NOT 'yes' -> SKIP. Labeled B but the checklist
                   disagrees, so drop it rather than keep a mislabeled row.
    """
    rows = []
    already_done = load_existing_titles(out_path, cls) if out_path else set()
    if already_done:
        print(f"[{cls}] checkpoint found: {len(already_done)} already collected in {out_path}, resuming")

    out_f = open(out_path, "a", encoding="utf-8") if out_path else None
    collected_count = len(already_done)

    for i, article_title in enumerate(article_titles):
        if collected_count >= n:
            print(f"[{cls}] target of {n} reached, stopping")
            break

        if article_title in already_done:
            continue

        print(f"[{cls}] ({i+1}/{len(article_titles)}) checking: {article_title}")
        time.sleep(delay)  # throttle every candidate, not just accepted rows

        talk_title = f"Talk:{article_title}"
        wikitext, ts = get_wikitext(talk_title)
        if wikitext is None:
            print(f"[{cls}] SKIP {article_title}: no talk page / no revision content")
            continue

        flags_full, conflict = extract_b_flags(wikitext)

        if cls == "C":
            if flags_full is None:
                print(f"[{cls}] SKIP {article_title}: no complete b1-b6 checklist found")
                continue
            flags = flags_full
            print(f"[{cls}] {article_title}: checklist found -> {flags}" + (" (CONFLICT)" if conflict else ""))
        else:  # cls == "B"
            if flags_full is not None:
                if any(v != "yes" for v in flags_full.values()):
                    print(f"[{cls}] SKIP {article_title}: checklist present but not all yes -> {flags_full}")
                    continue
                flags = flags_full
                print(f"[{cls}] {article_title}: checklist confirms all-yes")
            else:
                flags = {k: "yes" for k in B_KEYS}
                ts = None
                print(f"[{cls}] {article_title}: no checklist found, assuming all-yes")

        text = get_plaintext(article_title)
        if not text or len(text) < 200:
            print(f"[{cls}] SKIP {article_title}: text too short ({len(text) if text else 0} chars)")
            continue

        revid = get_latest_revid(article_title) if WIKIMEDIA_TOKEN else None
        liftwing_probs = query_liftwing(revid) if revid else None

        row = {
            "article_title": article_title,
            "overall_class": cls,
            "assessed_talk_timestamp": ts,
            "b1_referenced": flags["b1"],
            "b2_coverage": flags["b2"],
            "b3_structure": flags["b3"],
            "b4_grammar": flags["b4"],
            "b5_accessible": flags["b5"],
            "b6_supporting_materials": flags["b6"],
            "checklist_present": flags_full is not None,
            "banner_conflict": conflict,
            "revid": revid,
            "liftwing_probs": liftwing_probs,
            "text": text,
        }
        rows.append(row)
        collected_count += 1
        if out_f:
            out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
            out_f.flush()
        print(f"[{cls}] COLLECTED {article_title} ({collected_count}/{n}) -- checkpointed")

    if out_f:
        out_f.close()
    print(f"[{cls}] done: {collected_count} total collected ({len(rows)} new this run) out of {len(article_titles)} candidates")
    return rows


In [4]:
SEED = 230911091605040901 # WIKIPEDIA !!!

b_titles = load_quarry_titles("quarry/b_class_titles.csv", limit=N_PER_CLASS * 3, seed=SEED)
c_titles = load_quarry_titles("quarry/c_class_titles.csv", limit=N_PER_CLASS * 3, seed=SEED)
print(f"loaded {len(b_titles)} B-class candidate titles, {len(c_titles)} C-class candidate titles")

loaded 3000 B-class candidate titles, 3000 C-class candidate titles


In [5]:
all_rows = []
all_rows += build_dataset("B", b_titles, N_PER_CLASS, DELAY, out_path=OUT_PATH)
all_rows += build_dataset("C", c_titles, N_PER_CLASS, DELAY, out_path=OUT_PATH)

print(f"Done. {len(all_rows)} new rows collected this run, checkpointed incrementally to {OUT_PATH}")

[B] checkpoint found: 15 already collected in bclass_data.jsonl, resuming
[B] (16/3000) checking: Cell bank
    [http 429] Talk:Cell bank
    [429] rate-limited, waiting 24s (attempt 1/6)
    [http 200] Talk:Cell bank
[B] Cell bank: no checklist found, assuming all-yes
    [http 200] Cell bank
    [http 200] Cell bank
    [liftwing http 200] revid=1365911116
[B] COLLECTED Cell bank (16/1000) -- checkpointed
[B] (17/3000) checking: Dian Fossey
    [http 200] Talk:Dian Fossey
[B] Dian Fossey: no checklist found, assuming all-yes
    [http 200] Dian Fossey
    [http 200] Dian Fossey
    [liftwing http 200] revid=1371876232
[B] COLLECTED Dian Fossey (17/1000) -- checkpointed
[B] (18/3000) checking: International Unemployment Day
    [http 200] Talk:International Unemployment Day
[B] International Unemployment Day: no checklist found, assuming all-yes
    [http 200] International Unemployment Day
    [http 200] International Unemployment Day
    [liftwing http 200] revid=1370551202
[B] COLL

KeyboardInterrupt: 